In [ ]:
"""
即時剪貼簿去識別化 / 還原工具 (Microsoft Presidio)
- 複製含敏感資訊的文字     -> 自動遮蔽成 <ENTITY_n> 標籤並貼回剪貼簿
- 複製已含 <ENTITY_n> 標籤的文字 -> 不掃描，直接查表還原成原始敏感資訊並貼回剪貼簿
- 標籤對應存於 clipboard_map.txt (格式: 標籤\t原始內容)

安裝: pip install presidio-analyzer presidio-anonymizer pyperclip pywin32
      python -m spacy download en_core_web_lg
"""
import re
import time
import pyperclip
from presidio_analyzer import AnalyzerEngine

MAP_FILE = "clipboard_map.txt"
TAG_RE = re.compile(r"<([A-Z_]+)_(\d+)>")

analyzer = AnalyzerEngine()


# ---------- 過濾器區：想加/換偵測方式，寫一個回傳 [(start,end,entity_type),...] 的函式，append 進 FILTERS 即可 ----------
def presidio_filter(text):
    return [(r.start, r.end, r.entity_type) for r in analyzer.analyze(text=text, language="en")]


FILTERS = [presidio_filter]


# ---------- 對應表讀寫 ----------
def load_map():
    mapping = {}
    try:
        with open(MAP_FILE, encoding="utf-8") as f:
            for line in f:
                tag, _, val = line.rstrip("\n").partition("\t")
                if tag:
                    mapping[tag] = val
    except FileNotFoundError:
        pass
    return mapping


def save_map(mapping):
    with open(MAP_FILE, "w", encoding="utf-8") as f:
        for tag, val in mapping.items():
            f.write(f"{tag}\t{val}\n")


# ---------- 遮蔽 / 還原 ----------
def anonymize(text, mapping):
    matches = []
    for flt in FILTERS:
        matches += flt(text)
    if not matches:
        return None

    reverse = {v: k for k, v in mapping.items()}
    counters = {}
    for tag in mapping:
        t, i = TAG_RE.match(tag).groups()
        counters[t] = max(counters.get(t, 0), int(i))

    out = text
    for start, end, etype in sorted(matches, key=lambda m: m[0], reverse=True):
        val = text[start:end]
        tag = reverse.get(val)
        if not tag:
            counters[etype] = counters.get(etype, 0) + 1
            tag = f"<{etype}_{counters[etype]}>"
            mapping[tag] = val
            reverse[val] = tag
        out = out[:start] + tag + out[end:]

    save_map(mapping)
    return out


def deanonymize(text, mapping):
    return TAG_RE.sub(lambda m: mapping.get(m.group(0), m.group(0)), text)


# ---------- 主迴圈 ----------
def main():
    mapping = load_map()
    last = ""
    print("監控剪貼簿中... (Ctrl+C 結束)")
    while True:
        try:
            text = pyperclip.paste()
        except Exception:
            text = ""

        if text and text != last:
            new_text = deanonymize(text, mapping) if TAG_RE.search(text) else anonymize(text, mapping)
            if new_text and new_text != text:
                pyperclip.copy(new_text)
                text = new_text
            last = text

        time.sleep(0.5)


if __name__ == "__main__":
    main()

d:\2.programm2\github-star\presidio-research\venvpr\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


監控剪貼簿中... (Ctrl+C 結束)


KeyboardInterrupt: 

: 